# Data Ekplorasi

# Missing Value

In [113]:
import pandas as pd
from module.dataTransformer import combineData
from module.fetcher import fetchDataMysql, fetchDataPg

data_pg = fetchDataPg("SELECT petal_length, petal_width, species FROM iris_table")
data_my = fetchDataMysql("SELECT sepal_length, sepal_width FROM iris_table")

iris_df = combineData(data1=data_my, data2=data_pg)

print("Missing Value:", iris_df.isnull().sum(), end="\n")

Missing Value: sepal_length    0
sepal_width     0
petal_length    0
petal_width     0
species         0
dtype: int64


## Outlier

#### Mengecualikan kolom species

In [114]:
numeric_iris_data = iris_df[['sepal_length', 'sepal_width', 'petal_length', 'petal_width']]
numeric_iris_data

,sepal_length,sepal_width,petal_length,petal_width
0,5.10,3.50,1.40,0.20
1,4.90,3.00,1.40,0.20
2,4.70,3.20,1.30,0.20
3,4.60,3.10,1.50,0.20
4,5.00,3.60,1.40,0.20
...,...,...,...,...
145,6.70,3.00,5.20,2.30
146,6.30,2.50,5.00,1.90
147,6.50,3.00,5.20,2.00
148,6.20,3.40,5.40,2.30


### Outliers Detection Dengan (ABOD)

#### Menyiapkan model (ABOD)

In [60]:
from pycaret.anomaly import *


In [61]:
from pycaret.anomaly import *

s = setup(data=numeric_iris_data)

abod_model = create_model("abod", fraction=0.05)

df_abod = assign_model(abod_model)


,Description,Value
0,Session id,7307
1,Original data shape,"(150, 4)"
2,Transformed data shape,"(150, 123)"
3,Categorical features,4
4,Preprocess,True
5,Imputation type,simple
6,Numeric imputation,mean
7,Categorical imputation,mode
8,Maximum one-hot encoding,-1
9,Encoding method,None


#### Data di anggap Outliers

In [43]:
outliers = df_abod[df_abod["Anomaly"] == 1]
print(outliers.shape)
outliers

(8, 6)


,sepal_length,sepal_width,petal_length,petal_width,Anomaly,Anomaly_Score
8,4.40,2.90,1.40,0.20,1,-0.000000
33,5.50,4.20,1.40,0.20,1,-0.000000
77,6.70,3.00,5.00,1.70,1,-0.000185
108,6.70,2.50,5.80,1.80,1,-0.000123
124,6.70,3.30,5.70,2.10,1,-0.000185
129,7.20,3.00,5.80,1.60,1,-0.000185
130,7.40,2.80,6.10,1.90,1,-0.000123
145,6.70,3.00,5.20,2.30,1,-0.000195


#### Dataframe ABOD

jika nilai Anomaly 1 menunjukan data sebagai outlier

In [77]:
iris_abod = df_abod.merge(iris_df["species"], left_index=True, right_index=True)

iris_abod

,sepal_length,sepal_width,petal_length,petal_width,Anomaly,Anomaly_Score,species
0,5.10,3.50,1.40,0.20,0,-0.002695,Iris-setosa
1,4.90,3.00,1.40,0.20,0,-0.000938,Iris-setosa
2,4.70,3.20,1.30,0.20,0,-0.005625,Iris-setosa
3,4.60,3.10,1.50,0.20,0,-0.001406,Iris-setosa
4,5.00,3.60,1.40,0.20,0,-0.000820,Iris-setosa
...,...,...,...,...,...,...,...
145,6.70,3.00,5.20,2.30,1,-0.000195,Iris-virginica
146,6.30,2.50,5.00,1.90,0,-0.000271,Iris-virginica
147,6.50,3.00,5.20,2.00,0,-0.000938,Iris-virginica
148,6.20,3.40,5.40,2.30,0,-0.000316,Iris-virginica


```python
plot_model(abod_model, plot = 'tsne')
```


!["3D Plot"](i3d_plot_abod.png)

### Outliers Detection Dengan (iforest)

#### Menyiapkan Model KNN

In [103]:
s = setup(data=numeric_iris_data)
iforest_model = create_model('iforest', fraction=0.05)
iforest_df = assign_model(iforest_model)


,Description,Value
0,Session id,8482
1,Original data shape,"(150, 4)"
2,Transformed data shape,"(150, 123)"
3,Categorical features,4
4,Preprocess,True
5,Imputation type,simple
6,Numeric imputation,mean
7,Categorical imputation,mode
8,Maximum one-hot encoding,-1
9,Encoding method,None


#### Data Dianggap Outlier

In [104]:
outliers = iforest_df[iforest_df["Anomaly"] == 1]
print(outliers.shape)
outliers

(8, 6)


,sepal_length,sepal_width,petal_length,petal_width,Anomaly,Anomaly_Score
52,6.90,3.10,4.90,1.50,1,0.007413
54,6.50,2.80,4.60,1.50,1,0.011719
60,5.00,2.00,3.50,1.00,1,0.013863
97,6.20,2.90,4.30,1.30,1,0.007480
115,6.40,3.20,5.30,2.30,1,0.000342
117,7.70,3.80,6.70,2.20,1,0.002212
141,6.90,3.10,5.10,2.30,1,0.003465
148,6.20,3.40,5.40,2.30,1,0.005907


#### Dataframe iforest

In [105]:
iforest_df = iforest_df.merge(iris_df["species"], left_index=True, right_index=True)
iforest_df

,sepal_length,sepal_width,petal_length,petal_width,Anomaly,Anomaly_Score,species
0,5.10,3.50,1.40,0.20,0,-0.028641,Iris-setosa
1,4.90,3.00,1.40,0.20,0,-0.028799,Iris-setosa
2,4.70,3.20,1.30,0.20,0,-0.032128,Iris-setosa
3,4.60,3.10,1.50,0.20,0,-0.037257,Iris-setosa
4,5.00,3.60,1.40,0.20,0,-0.024745,Iris-setosa
...,...,...,...,...,...,...,...
145,6.70,3.00,5.20,2.30,0,-0.015738,Iris-virginica
146,6.30,2.50,5.00,1.90,0,-0.013565,Iris-virginica
147,6.50,3.00,5.20,2.00,0,-0.020859,Iris-virginica
148,6.20,3.40,5.40,2.30,1,0.005907,Iris-virginica


```python
plot_model(iforest_model, plot = 'tsne')
```

![plot outlier](i3d_plot_iforest.png)


### Outliers Detection dengan (LOF)

In [106]:
s = setup(data=numeric_iris_data)
lof_model = create_model("lof", fraction=0.05)

lof_df = assign_model(lof_model)

,Description,Value
0,Session id,4961
1,Original data shape,"(150, 4)"
2,Transformed data shape,"(150, 123)"
3,Categorical features,4
4,Preprocess,True
5,Imputation type,simple
6,Numeric imputation,mean
7,Categorical imputation,mode
8,Maximum one-hot encoding,-1
9,Encoding method,None


#### Data Outliers

In [109]:
outliers = lof_df[lof_df["Anomaly"] == 1]
print(outliers.shape)
outliers

(8, 6)


,sepal_length,sepal_width,petal_length,petal_width,Anomaly,Anomaly_Score
5,5.40,3.90,1.70,0.40,1,1.064198
57,4.90,2.40,3.30,1.00,1,1.061917
80,5.50,2.40,3.80,1.10,1,1.080816
81,5.50,2.40,3.70,1.00,1,1.063662
109,7.20,3.60,6.10,2.50,1,1.088615
117,7.70,3.80,6.70,2.20,1,1.069982
131,7.90,3.80,6.40,2.00,1,1.065950
144,6.70,3.30,5.70,2.50,1,1.054217


#### lof Dataframe

In [110]:
lof_df = lof_df.merge(iris_df["species"], left_index=True, right_index=True)
lof_df

,sepal_length,sepal_width,petal_length,petal_width,Anomaly,Anomaly_Score,species
0,5.10,3.50,1.40,0.20,0,0.997672,Iris-setosa
1,4.90,3.00,1.40,0.20,0,0.993906,Iris-setosa
2,4.70,3.20,1.30,0.20,0,1.001341,Iris-setosa
3,4.60,3.10,1.50,0.20,0,0.992439,Iris-setosa
4,5.00,3.60,1.40,0.20,0,0.993568,Iris-setosa
...,...,...,...,...,...,...,...
145,6.70,3.00,5.20,2.30,0,0.989725,Iris-virginica
146,6.30,2.50,5.00,1.90,0,1.023798,Iris-virginica
147,6.50,3.00,5.20,2.00,0,0.992359,Iris-virginica
148,6.20,3.40,5.40,2.30,0,1.003787,Iris-virginica


```python
plot_model(lof_model, plot='tsne')
```

![3d plot lof](i3d_plot_lof.png)

## Min, Max dan Average Setiap Kolom

Tabel berikut menampilkan nilai **maksimum (Max Value)**, **minimum (Min Value)**, dan **rata-rata (Average Value)** dari setiap fitur pada dataset Iris:  

![Min, Max Average Setiap Kolom](iris_min_ma.png)

### Interpretasi
- **Petal Length** memiliki rentang yang cukup lebar (1.00 – 6.90) dengan rata-rata 3.76.  
- **Petal Width** bervariasi dari 0.10 hingga 2.50, dengan rata-rata 1.20.  
- **Sepal Length** memiliki nilai rata-rata paling tinggi (5.84) dibandingkan fitur lain.  
- **Sepal Width** berada pada kisaran 2.00 – 4.40 dengan rata-rata 3.05.  

Dari ringkasan statistik ini dapat dilihat bahwa **petal** memiliki variasi lebih besar dibandingkan **sepal**, sehingga kemungkinan besar akan berperan penting dalam membedakan kelas spesies bunga Iris.

## Distribusi Kelas

Visualisasi berikut menunjukkan jumlah data pada masing-masing kelas:

- **Iris-setosa**: 50 data  
- **Iris-versicolor**: 50 data  
- **Iris-virginica**: 50 data  

Total dataset terdiri dari **150 data** yang terbagi **seimbang** antar ketiga kelas.

![Distribusi Kelas](new_species.png)

**Interpretasi:**
- Dataset **seimbang (balanced)** karena setiap kelas memiliki jumlah data yang sama.  
- Hal ini menguntungkan untuk proses klasifikasi, karena model tidak akan bias terhadap salah satu kelas.
